In [1]:
%%capture --no-stderr
%pip install -U --quiet langchain langgraph langchain_google_genai
%pip install -U --quiet saspy

ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'c:\\users\\talelbm\\appdata\\local\\anaconda3\\lib\\site-packages\\saspy\\java\\iomclient\\log4j-1.2-api-2.12.4.jar'
Consider using the `--user` option or check the permissions.



In [1]:
import math
from collections import deque
from typing import Optional, Dict, Any, List, Tuple
import os
import getpass
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers.openai_tools import JsonOutputToolsParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import chain as as_runnable
import pandas as pd
import re
from gradio_client import Client
from langgraph.graph import END, StateGraph, START
import google.generativeai as genai
genai.configure(api_key="YOUR_GOOGLE_API_KEY")

In [2]:
def _set_if_undefined(var: str) -> None:
    """Set environment variable if not already defined."""
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)

_set_if_undefined("GOOGLE_API_KEY")

GOOGLE_API_KEY ········


In [3]:
import json
with open('adress-to\meta_data.json', 'r', encoding='utf-8') as f:
    metadata_dict = json.load(f)

In [ ]:
import saspy
sas = saspy.SASsession(cfgname='iomwin', cfgfile='sascfg.py')

In [16]:
class MetadataVerifier:
    def __init__(self, model):
        self.model = model

    def verify_metadata(self, metadata: str, question: str, original_metadata: dict) -> tuple[bool, str]:
        verification_prompt = f"""Act as a critical data analyst reviewing metadata selection. 
Verify if the selected metadata is contains all the sufficient and necessary databases and columns for answering the query. do not go further than that.
if the database is sufficient to answer the query, approve it.

Question: {question}
Selected Metadata (to be analyzed carefully):
{metadata}

Complete Available Metadata (to be analyzed carefully):
{original_metadata}

Check for these specific issues:
1. Missing Essential Columns:
   - <necessary column names> must be present, if these exist, avoid criticism concerned with joining by key, these two columns are the keys to merge various dataframes
   - Columns needed for calculations mentioned in the query

2. Completeness Check:
   - All columns necessary for the query's calculations
   - All columns needed for filtering conditions
   - All columns needed for grouping or aggregation
   - All columns needed for the final output

3. Business Rules Verification:
   <redacted for privacy reasons>

Respond with a string containing:
    approved: boolean ("approved: True" or "approved : False"),
    issues: [list of specific issues found, strings with "..." format],
    missing columns: [list of essential missing columns, strings with "..." format],
    suggestions: [specific suggestions for improvement, strings with "..." format],
    criticism: "detailed explanation of why the metadata is or isn't optimal"
"""
        response = self.model.generate_content(verification_prompt)
        #print(response.text)
        return ("approved: True" in response.text, response.text)

class CachedMetadata:
    _instance = None

    def __init__(self, metadata_dict, question):
        self.metadata_dict = metadata_dict
        self.question = question
        self.necessary_metadata = self._generate_metadata()

    @classmethod
    def get_instance(cls, metadata_dict=None, question=None):
        if cls._instance is None:
            if metadata_dict is None or question is None:
                raise ValueError("Metadata dictionary and question must be provided for the first initialization.")
            cls._instance = CachedMetadata(metadata_dict, question)
        return cls._instance

    def __init__(self, question, metadata_dict):
        if CachedMetadata._instance is not None:
            raise Exception("This class is a singleton!")

        self.model_gemini = genai.GenerativeModel("gemini-2.0-flash-exp")
        self.question = question
        self.metadata_dict = metadata_dict
        self._metadata = None
        self._verifier = MetadataVerifier(self.model_gemini)
        self._generate_metadata()

    def _generate_metadata(self):
        prompt = f"""Act as an expert data analyst. Given a query and a database's metadata, identify the smallest set of databases 
        and columns necessary to answer the query accurately. Ensure that only the essential databases and columns are included, and 
        consider any required joins or relationships between tables. 
        Follow these specific rules:
        - give back a string in this format : 
        ""
        relevant_database1 : relevant_database1_description
            relevant_column11 : relevant_column11_type
                                relevant_column11_description
                                relevant_column11_values (if they exist)
            ...
        ""
        <Put clear instruction here dependant on ypur metadata>
        
        Provide the selected subset of metadata as a string, including only the necessary databases and their associated columns, without any additional explanations."
        Here is the question: {self.question}
        Here is the metadata: {self.metadata_dict}"""

        response = self.model_gemini.generate_content(prompt)
        self._metadata = response.text

        # Verify metadata immediately
        is_approved, verification_result = self._verifier.verify_metadata(
            self._metadata, self.question, self.metadata_dict
        )
        if not is_approved:
            self._regenerate_metadata(verification_result)
        return self._metadata

    def _regenerate_metadata(self, verification_result):
        regeneration_prompt = f"""Previous metadata selection was inadequate. Please regenerate the metadata selection addressing these issues:
        task:
        Act as an expert data analyst. Given a query and a database's metadata, identify the smallest set of databases 
        and columns necessary to answer the query accurately. Ensure that only the essential databases and columns are included, and 
        consider any required joins or relationships between tables. 
        Follow these specific rules:
        - give back a string in this format : 
        ""
        relevant_database1 : relevant_database1_description
            relevant_column11 : relevant_column11_type
                                relevant_column11_description
                                relevant_column11_values (if they exist)
            ...
        ""
        <Put clear instruction here dependant on ypur metadata>
        
        Provide the selected subset of metadata as a string, including only the necessary databases and their associated columns, without any additional explanations.
        The old response:
        {self._metadata}
        The full critique:
        {verification_result}

        Original question: {self.question}
        Available metadata: {self.metadata_dict}"""

        response = self.model_gemini.generate_content(regeneration_prompt)
        self._metadata = response.text

    def get_metadata(self):
        return self._metadata

In [6]:
def get_system_prompt(necessary_metadata) : 
    system_prompt = f"""
You are StarData, a powerful agentic AI coding agent.
You have basic understanding of actuary line of thinking, especially concerning automobile insurance.
you are an expert programmer in python and sql and SaS. respond by ONLY a sas query
- You are tasked in generating a sas query to answer data queries on a SaaS server
- do not give any html code. only a sas query with no explanations.
- you will be fed metadata of SaS databases. you need to use all the given metadata to answer the question.the entirety of the metadata is related to the question
- the metadata contains dataframes names and their general description, all their columns, their description and their type of each column, occasionally the columns values if they're categories. 
make use of this comprehensively and smartly to treat the query

Here is the metadata describing the tables and columns that you can use:
 {necessary_metadata}
          
steps to created the sas code:
1) read the question and understand it in the context of the metadata, rely each key term of the question to the metadata
2) perform the necessary operations to get the desired extraction

- Very important notes to keep in mind :
<Put clear instructions and notes here dependant on your metadata>


example:
question : "a query"

Very Important : this is an illustrative example that is highly dependant on the question and it shouldnt be taken verbatem as 
   
<Put the sas code that answers this question>
"""
    return system_prompt

In [7]:
def give_requirement_corrector_system_prompt(question,code,necessary_metadata, requirement_analysis = ""):
    base_prompt = f"""
You are a highly specialized data scientist and expert coder in SAS and SQL. Your task is to analyze a given query and a corresponding code snippet that is intended to meet the requirements of the query but falls short. Your objective is to correct and refine the code so that it accurately and faithfully answers the query.
To assist you in this task, you will be provided with metadata about the database. This metadata is crucial for understanding the structure, relationships, and constraints of the data, and it is all relevant to the query. Carefully study the metadata to ensure that your corrections align with the database's structure and the query's requirements.
Your final output should be a revised code snippet that fully addresses the query, leveraging the metadata to ensure accuracy and completeness.
---
Query: {question}
Code: {code}
Metadata: {necessary_metadata}
"""
    if requirement_analysis:
        base_prompt += f"\n\nPrevious analysis of requirements not being met:\n{requirement_analysis}\n\n"
        base_prompt += "Use this analysis to inform your suggestions for fixes and identification of missing requirements."
    
    return base_prompt

In [8]:
error_corrector_system_prompt = f"""-you are an expert in sql and SaS. respond by only sas and sql code. 
                  you're given a sas query along with the log resulting from sas entrepise guide after executing the sas code (which would be usually a one or a series of sql procedures)
                  you are tasked to read carefully the log and understand the errors occuring and then modify the query and give back the sas code
                  that you were given but with a sas code that avoid all the errors encountered in the log. thank you.
                  here are some recommendations for treating some common errors:
                  - make the databases' names short to avoid possible errors
                  - ERROR: Column column_name could not be found in the table/view identified with the correlation name table_name : 
                        1) Verify the exact column names in the source table using PROC CONTENTS or by checking the data dictionary
                        2) Ensure the column name is spelled correctly and matches the case exactly
                        3) Check if the column has been renamed or dropped in a previous data step
                        4) Confirm that the table is properly libref'd and accessible
                        5) If using a join or merged dataset, verify the column exists in the specific dataset you're referencing
                        6) Use the DICTIONARY.COLUMNS view to cross-check column names across different tables
                  - ERROR: Column 10 from the first contributor of UNION ALL is not the same type as its counterpart from the second: 
                        1) Explicitly cast columns to a common data type before the UNION ALL
                        2) Use the PUT() function to convert numeric to character, or INPUT() function to convert character to numeric
                        3) Ensure all columns have consistent lengths and types across all datasets being unioned
                        4) Check data types using PROC CONTENTS on each dataset before the UNION ALL
                        5) Use LENGTH statements to standardize column types before combining datasets
                        6) If mixing numeric and character types, create explicit conversion rules
                        7) Consider using PROC SQL instead of UNION ALL if type conversions are complex
                  - ERROR: Numeric expression requires a numeric format. : 
                        1) Use the INPUT() function to convert character variables to numeric
                        2) Verify the current format of the variable using PROC CONTENTS
                        3) Remove any non-numeric characters before conversion (e.g., commas, currency symbols)
                        4) Use appropriate informat to handle different numeric representations
                        5) Check for leading/trailing spaces that might prevent numeric conversion
                        6) Use the COMPRESS() function to remove non-numeric characters
                        7) Explicitly define the numeric format using FORMAT or INFORMAT statements
                        8) Validate data type and format compatibility before performing numeric operations
                UNION ALL BEST PRACTICES:
                    1. NEVER use SELECT *, instruction : when seeing Select * from database replace with select <necessary columns> from database
                    2. ALWAYS explicitly list columns
                    3. Ensure type consistency across tables
                    4. Handle null values proactively
                    5. Add year/source identifiers
                    6. Preprocess tables before union
                    7. Use metadata-driven approaches
                    8. Verify column structures before combining
                    Explicit Column Selection

Never use select *
Always list columns explicitly
Ensure IDENTICAL column list across all UNION ALL contributors
Use CAST() to standardize types
Use COALESCE() to handle potential null values


Preprocessing Techniques

Before UNION ALL, normalize tables

sasCopy/* Pre-UNION ALL Normalization Example */
data normalized_2019;
  set veh_sinistre_2019;
  format col1-col5 best.; /* Standardize numeric formats */
  year = '2019';           /* Add year identifier */
run;

Type Conversion Strategies

Use INPUT() for character to numeric conversions
Use PUT() for numeric to character conversions
Always explicitly handle type conversions


Error Prevention Checklist

Verify column names exactly
Check column types before UNION ALL
Use PROC CONTENTS to inspect table structures
Create a column mapping matrix


Advanced Technique: Metadata-Driven Unions

sasCopy/* Dynamic UNION ALL using metadata */
%macro dynamic_union(years=2019 2020 2021 2022 2023 2024);
    %let first_table = %scan(&years, 1);
    
    /* Initial table structure reference */
    proc sql noprint;
        create table union_structure as
        select * from veh_sinistre_&first_table. (obs=0);
    quit;

    data veh_sinistres_all;
        set 
        %do year = 1 %to %sysfunc(countw(&years));
            veh_sinistre_%scan(&years, &year)
        %end;;
    run;
%mend;

FORBIDDEN PATTERNS:
- select * from table
- Implicit type conversions
- Inconsistent column lists

RECOMMENDED PATTERN:
select 
    explicitly_listed_columns,
    cast_to_consistent_types,
    add_source_identifiers
from each_source_table"""

In [9]:
# Base Models for Reflection
class ErrorReflection(BaseModel):
    error_count: int = Field(description="Number of errors in SAS log")
    error_fixes: List[str] = Field(description="Specific fixes for each error")
    found_solution: bool = Field(description="Whether all errors are resolved")

    def as_message(self) -> str:
        return f"Errors: {self.error_count}, Fixes needed: {len(self.error_fixes)}, Solved: {self.found_solution}"

class RequirementReflection(BaseModel):
    missing_requirements: List[str] = Field(description="Requirements not met")
    requirement_fixes: List[str] = Field(description="Changes needed to meet requirements")
    requirements_met: bool = Field(description="Whether all requirements are met")

    def as_message(self) -> str:
        return f"Missing reqs: {len(self.missing_requirements)}, Fixes needed: {len(self.requirement_fixes)}, Met: {self.requirements_met}"

In [10]:
class Reflector:
    def __init__(self, llm, client,necessary_metadata):
        self.llm = llm
        self.client = client
        self.necessary_metadata = necessary_metadata
        self._cached_responses = {}
        
    def get_reflections(self, query: str, sas_code: str, log: str) -> Tuple[ErrorReflection, RequirementReflection]:
        cache_key = (query, sas_code, log)
        if cache_key in self._cached_responses:
            return self._cached_responses[cache_key]
            
        error_refl = self._get_error_reflection(query, sas_code, log)
        req_refl = self._get_requirement_reflection(query, sas_code, self.necessary_metadata)
        
        result = (error_refl, req_refl)
        self._cached_responses[cache_key] = result
        return result
    
    def _get_error_reflection(self, query: str, sas_code: str, log: str) -> ErrorReflection:
        error_count = len(re.findall(r'ERROR', str(log))) if log else 0
        if error_count == 0:
            return ErrorReflection(error_count=0, error_fixes=[], found_solution=True)
            
        prompt = ChatPromptTemplate.from_messages([
            ("system", error_corrector_system_prompt),
            ("user", f"Query: {query}\nCode:\n```sas\n{sas_code}\n```\nLog:\n{log}")
        ])
        
        reflections = (prompt | self.llm | JsonOutputToolsParser()).invoke({
            'query': query,
            'sas_code': sas_code,
            'log': log
        })
        
        error_fixes = []
        for reflection in reflections:
            if isinstance(reflection, dict) and 'error_fixes' in reflection:
                error_fixes.extend(reflection['error_fixes'])
                
        return ErrorReflection(
            error_count=error_count,
            error_fixes=error_fixes,
            found_solution=error_count == 0
        )
    
    def _get_requirement_reflection(self, query: str, sas_code: str, necessary_metadata : str) -> RequirementReflection:
        
        # First, check requirements and get detailed analysis if they're not met
        meets_requirements, requirement_analysis = self._check_requirements(query, sas_code,necessary_metadata)
        
        # Only proceed with LLM call if requirements aren't met or we need additional analysis
        if not meets_requirements:
            # Include the requirement analysis in the system prompt
            enhanced_prompt = ChatPromptTemplate.from_messages([
                ("system", give_requirement_corrector_system_prompt(
                    query, sas_code, self.necessary_metadata, requirement_analysis)),
                ("user", "Query: {query}\nCode:\n```sas\n{sas_code}\n```")
            ])
            
            reflections = (enhanced_prompt | self.llm | JsonOutputToolsParser()).invoke({
                'query': query,
                'sas_code': sas_code
            })
            
            missing_requirements = []
            requirement_fixes = []
            for reflection in reflections:
                if isinstance(reflection, dict):
                    missing_requirements.extend(reflection.get('missing_requirements', []))
                    requirement_fixes.extend(reflection.get('requirement_fixes', []))
        else:
            missing_requirements = []
            requirement_fixes = []
        
        return RequirementReflection(
            missing_requirements=missing_requirements,
            requirement_fixes=requirement_fixes,
            requirements_met=meets_requirements
        )
        
    def _check_requirements(self, query: str, sas_code: str, necessary_metadata : str) -> Tuple[bool, str]:
        cache_key = (query, sas_code)
        if cache_key in self._cached_responses:
            cached_result = self._cached_responses[cache_key]
            # If cached result is a boolean, return with empty analysis
            if isinstance(cached_result, bool):
                return cached_result, ""
            return cached_result
            
        prompt = f"""
        Original Query: {query}
        Generated SAS Code:
        ```sas
        {sas_code}
        ```
        metadata : {necessary_metadata}
        Does the SAS code fulfill all requirements stated in the original query?
        Answer with 'True' if everything is generally satisfying to the average user, only give back "False" if there flagrant error of totally not understanding the query.
        default answer is 'True'
        """
        
        result = self.client.predict(query=prompt, api_name="/generation_code")
        
        if result[0] == "True":
            meets_requirements = True
            analysis = ""
        else:
            meets_requirements = False
            analysis = result[0]
        
        cache_result = (meets_requirements, analysis)
        self._cached_responses[cache_key] = cache_result
        return cache_result

In [11]:
class CodeCorrector:
    def __init__(self, llm, necessary_metadata):
        self.llm = llm
        self.necessary_metadata = necessary_metadata
        self._correction_cache = {}
        
    def correct_code(self, query: str, sas_code: str, log: str, 
                    error_refl: ErrorReflection, req_refl: RequirementReflection) -> str:
        cache_key = (query, sas_code, log, 
                    tuple(error_refl.error_fixes), 
                    tuple(req_refl.requirement_fixes))
                    
        if cache_key in self._correction_cache:
            return self._correction_cache[cache_key]
            
        corrected_code = sas_code
        
        # Handle error fixes
        if error_refl.error_count > 0:
            prompt = ChatPromptTemplate.from_messages([
                ("system", error_corrector_system_prompt),
                ("user", f"Query: {query}\nCode:\n```sas\n{sas_code}\n```\nLog:\n{log}\nFixes:\n{error_refl.error_fixes}")
            ])
            response = (prompt | self.llm).invoke({
                'query': query,
                'sas_code': sas_code,
                'log': log,
                'error_fixes': error_refl.error_fixes
            })
            corrected_code = self._extract_code(response.content)
            
        # Handle requirement fixes
        if not req_refl.requirements_met:
            prompt = ChatPromptTemplate.from_messages([
                ("system", give_requirement_corrector_system_prompt(query, corrected_code, self.necessary_metadata)),
                ("user", f"Query: {query}\nCode:\n```sas\n{corrected_code}\n```\nFixes:\n{req_refl.requirement_fixes}")
            ])
            response = (prompt | self.llm).invoke({
                'query': query,
                'sas_code': corrected_code,
                'requirement_fixes': req_refl.requirement_fixes
            })
            corrected_code = self._extract_code(response.content)
            
        self._correction_cache[cache_key] = corrected_code
        return corrected_code
        
    def _extract_code(self, text: str) -> str:
        if "```" in text:
            start = text.find("```sas") + len("```sas")
            end = text.find("```", start)
            if start > -1 and end > -1:
                return text[start:end].strip()
        return text

In [12]:
# Node and TreeState implementations
class Node:
    def __init__(
        self,
        sas_code: str,
        error_reflection: ErrorReflection,
        req_reflection: RequirementReflection,
        parent: Optional["Node"] = None,
        query: Optional[str] = None,
    ):
        self.sas_code = sas_code
        self.parent = parent
        self.children = []
        self.value = 0
        self.visits = 0
        self.error_reflection = error_reflection
        self.req_reflection = req_reflection
        self.depth = parent.depth + 1 if parent is not None else 1
        self._is_solved = (
            error_reflection.found_solution and 
            req_reflection.requirements_met
        ) if error_reflection and req_reflection else False
        self.query = query if query is not None else (parent.query if parent is not None else None)
        
        if self._is_solved:
            self._mark_tree_as_solved()
        if error_reflection and req_reflection:
            self.backpropagate(self._calculate_score())

    def _calculate_score(self) -> float:
        max_errors = 10
        error_score = max(0, (max_errors - self.error_reflection.error_count) / max_errors)
        requirement_score = 1.0 if self.req_reflection.requirements_met else 0.5
        return (error_score + requirement_score) / 2

    def upper_confidence_bound(self, exploration_weight: float = 1.0) -> float:
        if self.parent is None:
            raise ValueError("Cannot calculate UCB for root node")
        if self.visits == 0:
            return float('inf')
        exploitation = self.value / self.visits
        exploration = math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration_weight * exploration

    def backpropagate(self, reward: float) -> None:
        node = self
        while node:
            node.visits += 1
            node.value = (node.value * (node.visits - 1) + reward) / node.visits
            node = node.parent

    def _mark_tree_as_solved(self) -> None:
        parent = self.parent
        while parent:
            parent._is_solved = True
            parent = parent.parent

    @property
    def is_solved(self) -> bool:
        return self._is_solved

    def get_best_solution(self) -> "Node":
        all_nodes = [self] + self._get_all_children()
        return max(
            all_nodes,
            key=lambda node: int(node.is_solved) * node.value
        )

    def _get_all_children(self) -> List["Node"]:
        all_nodes = []
        queue = deque(self.children)
        while queue:
            node = queue.popleft()
            all_nodes.append(node)
            queue.extend(node.children)
        return all_nodes

    @property
    def height(self) -> int:
        if self.children:
            return 1 + max([child.height for child in self.children])
        return 1

class TreeState(TypedDict):
    root: Optional[Node]
    input: str
    terminate: bool

In [17]:
# Graph Functions
from langchain_core.messages import HumanMessage
def generate_initial_response(state: TreeState, necessary_metadata: str) -> dict:
    """Generate initial SAS code response with cached metadata, learning from previous attempts."""
    previous_attempts = []
    
    while True:
        # Start with the base context as a human message
        messages = [
            HumanMessage(content=get_system_prompt(necessary_metadata))
        ]
        
        # Add previous attempts feedback if any
        if previous_attempts:
            feedback_content = "Previous attempts and their issues:\n\n"
            for i, attempt in enumerate(previous_attempts, 1):
                feedback_content += f"Attempt {i}:\n```sas\n{attempt['code']}\n```\n"
                feedback_content += "Issues found:\n"
                
                if attempt['error_refl'].error_count > 0:
                    feedback_content += "Errors:\n"
                    for fix in attempt['error_refl'].error_fixes:
                        feedback_content += f"- {fix}\n"
                
                if not attempt['req_refl'].requirements_met:
                    feedback_content += "Missing Requirements:\n"
                    for req in attempt['req_refl'].missing_requirements:
                        feedback_content += f"- {req}\n"
                    feedback_content += "Suggested Fixes:\n"
                    for fix in attempt['req_refl'].requirement_fixes:
                        feedback_content += f"- {fix}\n"
                
                feedback_content += "\n"
            
            messages.append(HumanMessage(content=feedback_content))
            messages.append(HumanMessage(content="Please generate a new solution that addresses all the issues mentioned above."))
        
        # Add the actual question
        messages.append(HumanMessage(content=f"Please write SAS code to answer this question: {state['input']}"))
        
        initial_prompt = ChatPromptTemplate.from_messages(messages)
        
        # Generate new response
        response = (initial_prompt | llm).invoke({"input": state["input"]})
        sas_code = code_corrector._extract_code(response.content)
        #print("\n initial sas code ", sas_code)
        log = execute_sas_code(sas_code)
        #print("\n initial log :",log)
        
        # Get reflections
        error_refl, req_refl = reflector.get_reflections(state["input"], sas_code, log)
        #print("\n error reflection : ",error_refl)
        #print("\n requirement reflection : ",req_refl) 
        # Check if solution is acceptable
        if req_refl.requirements_met:
            root = Node(sas_code, error_refl, req_refl, query=state["input"])
            return {**state, "root": root, "terminate": root.is_solved}
        
        # Store attempt for next iteration
        previous_attempts.append({
            'code': sas_code,
            'error_refl': error_refl,
            'req_refl': req_refl
        })
        
        print(f"Regenerating initial response (attempt {len(previous_attempts)})...")
        
def expand(state: TreeState, config: Dict[str, Any]) -> dict:
    """Expand tree with new candidates using cached components."""
    root = state["root"]
    best_candidate = select(root)
    
    # Get current state
    log = execute_sas_code(best_candidate.sas_code)
    error_refl, req_refl = reflector.get_reflections(
        best_candidate.query,
        best_candidate.sas_code,
        log
    )
    
    # Generate correction
    corrected_code = code_corrector.correct_code(
        best_candidate.query,
        best_candidate.sas_code,
        log,
        error_refl,
        req_refl
    )
    
    # Create new node if requirements are met
    new_log = execute_sas_code(corrected_code)
    new_error_refl, new_req_refl = reflector.get_reflections(
        best_candidate.query,
        corrected_code,
        new_log
    )
    
    if new_req_refl.requirements_met:
        child_node = Node(
            corrected_code,
            new_error_refl,
            new_req_refl,
            parent=best_candidate,
            query=best_candidate.query
        )
        best_candidate.children.append(child_node)
        terminate = child_node.is_solved
    else:
        print("Requirements not met, skipping expansion")
        terminate = False
    
    return {**state, "terminate": terminate}

def select(root: Node) -> Node:
    """Select best node using UCB."""
    if not root.children:
        return root
    node = root
    while node.children:
        node = max(node.children, key=lambda child: child.upper_confidence_bound())
    return node

def should_loop(state: TreeState) -> str:
    """Determine whether to continue search."""
    return END if state["terminate"] else "expand"

def execute_sas_code(sas_code: str) -> str:
    """Execute SAS code and return log."""
    sas.submit(sas_code)
    return sas.lastlog()

def extract_data(sas_code: str) -> None:
    """Execute SAS code and save results to Excel."""
    sas.submit(sas_code)
    df_name_result = client.predict(
        query=f"Extract final dataframe name from, dont include 'WORK.', just the raw name:\n{sas_code}", 
        api_name="/generation_code"
    )
    
    if df_name_result:
        df_name = df_name_result[0]
        df = sas.sd2df(df_name)
        if not df.empty:
            excel_file = f"{df_name}.xlsx"
            df.to_excel(excel_file, index=False)
            print(f"Data extracted to {excel_file}")

In [14]:
# Build Graph
def graph_builder(necessary_metadata):
    builder = StateGraph(TreeState)
    from functools import partial
    # Create a partial function that includes the necessary_metadata
    generate_initial_response_with_metadata = partial(generate_initial_response, necessary_metadata=necessary_metadata)
    builder.add_node("start", generate_initial_response_with_metadata)
    builder.add_node("expand", expand)
    builder.add_edge(START, "start")
    builder.add_conditional_edges("start", should_loop, {"expand": "expand", END: END})
    builder.add_conditional_edges("expand", should_loop, {"expand": "expand", END: END})
    return builder

def run_sas_agent(question: str,necessary_metadata:str, max_steps: int = 20) -> None:
    """Run SAS agent to generate and optimize code."""
    step_count = 0
    graph = graph_builder(necessary_metadata).compile()
    for step in graph.stream({"input": question, "root": None, "terminate": False}):
        step_name, step_state = next(iter(step.items()))
        if not step_state['terminate']:
            print(f"Step: {step_name}")
            if step_state['root']:
                print(f"Tree height: {step_state['root'].height}")
            step_count += 1
            if step_count >= max_steps:
                print("Max steps reached")
                break
        else:
            final_node = step_state["root"].get_best_solution()
            if final_node and final_node.is_solved:
                print("Final SAS Code:")
                print(final_node.sas_code)
                extract_data(final_node.sas_code)
            else:
                print("No solution found")

In [ ]:
question = """..."""
llm = ChatGoogleGenerativeAI(model="gemini-exp-1206", convert_system_message_to_human=True, temperature = 0.5)
client = Client("Qwen/Qwen2.5-Coder-Artifacts")
necessary_metadata__ = CachedMetadata.get_instance(metadata_dict, question)._metadata
print("\n necessary metadata : ", necessary_metadata__)
reflector = Reflector(llm, client,necessary_metadata__)
code_corrector = CodeCorrector(llm, necessary_metadata__)
    
# Example usage
run_sas_agent(question,necessary_metadata__)